## **Install Necessary Libraries**

#### Installing BeautifulSoup

In [2]:
pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


#### Installing Selenium

In [3]:
pip install selenium

#### Intalling WebDriver Manager

In [4]:
pip install webdriver-manager

Note: you may need to restart the kernel to use updated packages.


## **Execution**

The below code is a web scraping script made using Selenium and BeautifulSoup to extract information about Samsung TVs which are available for order as of this code is typed.

WebDriver sets up a Chrome WebDriver using ChromeDriverManager to handle Chrome browser automation. The target URL is "https://www.samsung.com/ca/tvs/all-tvs/?available-to-order".

Consent Banner Handling is done to wait for the Cookie Preferences Consent Banner to appear and then accept it by locating the banner element and clicking the "Accept All" button.

The 'View More' button on the page is identified and clicked using Selenium's WebDriverWait and expected conditions. 

Then, the webpage is scrolled to the bottom of the page to load all the TVs dynamically, waiting for the page to expand as it loads more content.

After loading all the TVs, the page source is retrieved and is parsed using BeautifulSoup, and all the TVs are found by searching for 'div' elements with a specific class.

For each TV, relevant information such as **Product Name**, **Product Price**, **SKU Code**, **Financing Option**, **Rating**, and **Number of Ratings**, are extracted by navigating to the individual TV's URL.

A Pandas DataFrame is then created and the extracted information about each TV is stored in the DataFrame.

Finally, WebDriver is quit, closing the browser.

----------
This python notebook is completlely coded and written by **Niraj Pradipkumar Patil** with help of some online resources mentioned below.

*[These resources were used to implement:*

***WebDriverWait:** https://www.selenium.dev/selenium/docs/api/py/webdriver_support/selenium.webdriver.support.wait.html*

***Expected Conditions:** https://www.selenium.dev/selenium/docs/api/py/webdriver_support/selenium.webdriver.support.expected_conditions.html*

***By Module:** https://www.selenium.dev/selenium/docs/api/py/webdriver/selenium.webdriver.common.by.html#module-selenium.webdriver.common.by]*

*{Note: Usually all the websites change their structure over time, so the below script might need adjustments if the structure of the target website changes.}*

In [14]:
# Importing Necessary Libraries

import time
import pandas as pd
from selenium import webdriver
from bs4 import BeautifulSoup
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

opts = Options()

# Set up Selenium webdriver
url = "https://www.samsung.com/ca/tvs/all-tvs/?available-to-order"
driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()), options = opts)
#driver.maximize_window()
driver.get(url)

# Wait for the Cookie Preferences Consent Banner to Pop-up and then Accept it
consent_banner = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.ID, "truste-consent-track")))
consent_banner.find_element(By.XPATH, "//button[text()='Accept All']").click()

# Identify & Click 'View More' button
button = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//a[@class='cta cta--outlined cta--black cta--icon js-pfv2-view-more-cta']")))
button.click()

# Scroll to the bottom of the page to load all the TVs
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

# Get the page source after loading all TVs
page_source = driver.page_source

# Parse the HTML using BeautifulSoup
soup = BeautifulSoup(page_source, 'html.parser')
#print(soup)

# Find all the TVs
productlist = soup.find_all('div', {'class': "pd03-product-card pd03-product-card--horizontal"})
print('Number of TV Models Available to Order Now:', len(productlist))

# Extract Product URLs for each TV
productlinks = []
baseurl = "https://www.samsung.com"    # Base URL will be further used to store the complete URLs of the TVs
for item in productlist:
    for link in item.find_all('a', class_= "pd03-product-card__product-image-link", href = True):
        productlinks.append(baseurl + link['href'])    # Concatenating the product href to the base URL to store complete and accessible URLs for each product
#print(len(productlinks))

tvlist = []
for link in productlinks:
    product_url = link
    driver.get(link)
    link = driver.page_source
    productdata = BeautifulSoup(link, 'html.parser')

    name = productdata.find('h2', class_ = 'pd-info__title').text.strip()
    price = productdata.find('span', class_ = 'pd-buying-price__new-price-currency').text.strip()
    sku_code = productdata.find('span', class_ = 'pd-info__sku-code').text.strip()
    m = len(price)
    fin_opt = productdata.find('div', class_ = 'pd-buying-price__new-price').text.strip()[:-(m+4)]
    if fin_opt == '':
        fin_opt = 'Not Applicable'
    rating = productdata.find('strong', class_ = 'rating__point').text.strip()[19:]
    rating_count = productdata.find('em', class_ = 'rating__review-count').text.strip()[20:-1]

    tv = {
        'Product Name': name,
        'Product Price': price,
        'Financing Option': fin_opt,
        'Product Rating': rating,
        'Number of Ratings': rating_count,
        'SKU Code': sku_code,
        'Product URL': product_url
    }
    tvlist.append(tv)

# Create a Pandas DataFrame & Store the Scrapped Data in it
df = pd.DataFrame(tvlist)

# Close the Webdriver
driver.quit()

Number of TV Models Available to Order Now: 22


In [15]:
# Display the Scrapped Data in a DataFrame
df

,Product Name,Product Price,Financing Option,Product Rating,Number of Ratings,SKU Code,Product URL
0,75” The Frame Art Mode LS03B,"$3,499.99",From $145.83/mo for 24 mos,4.4,2217,QN75LS03BAFXZC,https://www.samsung.com/ca/lifestyle-tvs/the-f...
1,"43"" Crystal UHD 4K Smart TV CU7000",$429.99,From $15.97/mo for 36 mos,4.5,1685,UN43CU7000FXZC,https://www.samsung.com/ca/tvs/uhd-4k-tv/cryst...
2,"55"" OLED 4K Smart TV S90C","$1,899.99",From $52.78/mo for 36 mos,4.8,1528,QN55S90CAFXZC,https://www.samsung.com/ca/tvs/oled-tv/s90c-55...
3,"43"" Crystal UHD 4K Smart TV CU8000",$649.99,From $24.14/mo for 36 mos,4.6,565,UN43CU8000FXZC,https://www.samsung.com/ca/tvs/uhd-4k-tv/cu800...
4,"43"" QLED 4K Q60C",$599.99,From $22.28/mo for 36 mos,4.7,1149,QN43Q60CAFXZC,https://www.samsung.com/ca/tvs/qled-tv/q60c-43...
5,43” Crystal UHD 4K Smart TV Powered by Tizen™ ...,$499.99,From $18.57/mo for 36 mos,4.4,651,UN43TU690TFXZC,https://www.samsung.com/ca/tvs/uhd-4k-tv/tu690...
6,"55"" OLED 4K Smart TV S95C","$2,499.99",From $69.44/mo for 36 mos,4.8,1206,QN55S95CAFXZC,https://www.samsung.com/ca/tvs/oled-tv/s95c-55...
7,"55"" Neo QLED 4K QN85C","$1,399.99",From $38.89/mo for 36 mos,4.8,822,QN55QN85CAFXZC,https://www.samsung.com/ca/tvs/qled-tv/qn85c-5...
8,"32"" Full HD Smart TV N5300",$299.99,From $11.14/mo for 36 mos,3.4,222,UN32N5300AFXZC,https://www.samsung.com/ca/tvs/full-hd-tv/n530...
9,"32"" HD Smart TV M4500B",$229.99,From $8.54/mo for 36 mos,3.8,187,UN32M4500BFXZC,https://www.samsung.com/ca/tvs/hd-tv/m4500b-32...


In [22]:
# Exporting the Scrapped Data to a '.csv' file
df.to_csv(r"Samsung_TV_Scrapped_Data.csv")

In [24]:
# Exporting the Scrapped Data to a '.xlsx' file
df.to_excel(r"Samsung_TV_Scrapped_Data.xlsx")